# Entity-Relation Classification Pipeline

Classifies extracted `(software/hardware, vulnerability)` pairs into:

| Type | Description |
|------|-------------|
| **A** | CVE present in vulnerability field **and** the pair is found in CISA KEV or CVE JSON descriptions |
| **B** | CVE present but pair is **not** found in either datasource (new/unknown relation) |
| **C** | No CVE in text, but NL vulnerability mention **maps to a CVE** via CISA/CVE text matching |
| **D** | No CVE, NL mention **maps to a CWE** |
| **E** | Remainder — unmatched natural language vulnerability mention |

**Input**: `software_relations_per_article.csv` (with `url` column)  
**Reference data**: `cve.json`, `known_exploited_vulnerabilities.csv`, `cwe.json`  
**Output**: `classification_types_in_json.json`

## 1. Imports & Config

In [1]:
import re
import json
import pandas as pd
from collections import defaultdict
from typing import Any, Dict, List, Optional, Set, Tuple, Union
from tqdm.auto import tqdm

try:
    from rapidfuzz import fuzz
    HAVE_RAPIDFUZZ = True
    print("rapidfuzz available")
except ImportError:
    HAVE_RAPIDFUZZ = False
    print("rapidfuzz not found — fuzzy matching disabled")

# ── File paths ──────────────────────────────────────────────────────────────
CSV_PATH        = "software_relations_per_article.csv"
CVE_JSON_PATH   = "cve.json"
CISA_CSV_PATH   = "known_exploited_vulnerabilities.csv"
CWE_JSON_PATH   = "cwe.json"
OUTPUT_PATH     = "classification_types_in_json.json"

rapidfuzz available


## 2. Shared Utilities

All regex patterns and helper functions defined once and reused throughout.

In [2]:
# ── Regex patterns ───────────────────────────────────────────────────────────

_CVE_RE = re.compile(
    r"\bCVE[\-\u2010-\u2015\u2212 ]?(\d{4})[\-\u2010-\u2015\u2212 ]?(\d+)\b",
    re.IGNORECASE,
)
_CWE_RE = re.compile(r"\bCWE-\d+\b", re.IGNORECASE)
_PAT_NON_ALNUM = re.compile(r"[^a-z0-9]+", re.IGNORECASE)
_PAT_SPACES = re.compile(r"\s+")
_TRAILING_JUNK = re.compile(r"[\s,;:\-]+$")
_UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

_VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?   # 1.2.3, 6.7U3, 3.1a
        |
        \d+(?:[A-Za-z]\d+)+                      # 6U3 style
        |
        r\d+p\d+                                 # r43p0 style
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

_RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + _VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + _VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + _VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + _VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + _VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + _VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

_SW_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |release\s+train|update|instances?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE,
)

_HW_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |update|instances?
        |devices?|appliances?|routers?|firewalls?|gateways?|switches?|servers?
        |units?|models?|platforms?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE,
)

# Type C stop tokens (very generic — skip when building inverted index)
_STOP_TOKENS: Set[str] = {
    "the","and","with","for","from","into","over","under","that","this",
    "vulnerability","vulnerabilities","issue","issues","plugin","plugins",
    "product","products","allows","could","may","might","lead","leading",
    "contain","contains","containing","via","using","used","improper",
    "insufficient","validation","check","checks","bypass","denial","service",
    "remote","code","execution","arbitrary","information","disclosure",
    "local","attack","attacker","attackers","crafted","malformed","request",
    "requests","response","unauthenticated","authenticated","privilege",
    "privileges","elevation","escalation","xss","sql","injection","rce",
}


# ── Text helpers ─────────────────────────────────────────────────────────────

def norm_text(s: str) -> str:
    """Lowercase, replace non-alphanumeric with spaces, collapse whitespace."""
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.lower()
    s = _PAT_NON_ALNUM.sub(" ", s)
    return _PAT_SPACES.sub(" ", s).strip()


def tokenize(s: str, min_len: int = 4, stop: Set[str] = _STOP_TOKENS) -> List[str]:
    return [t for t in norm_text(s).split() if len(t) >= min_len and t not in stop]


def token_subset(a: str, b: str) -> bool:
    """Are all tokens in `a` contained in `b`? (a ⊆ b)"""
    A = set(norm_text(a).split())
    B = set(norm_text(b).split())
    return bool(A) and A.issubset(B)


def fuzzy_match(a: str, b: str, threshold: int = 85) -> bool:
    if not HAVE_RAPIDFUZZ:
        return False
    return fuzz.partial_ratio(a, b) >= threshold


def _dedupe(items: List[str]) -> List[str]:
    seen: Set[str] = set()
    return [x for x in items if not (x in seen or seen.add(x))]


# ── CVE / CWE helpers ────────────────────────────────────────────────────────

def normalize_cve(year: str, num: str) -> str:
    return f"CVE-{year}-{num}".upper()


def find_all_cves(text: str) -> List[str]:
    return [normalize_cve(m.group(1), m.group(2)) for m in _CVE_RE.finditer(text)]


def extract_cve(text: str) -> Optional[str]:
    """Return first CVE ID found in text, or None."""
    m = _CVE_RE.search(text)
    return normalize_cve(m.group(1), m.group(2)) if m else None


def extract_cwe(text: str) -> Optional[str]:
    m = _CWE_RE.search(text)
    return m.group(0).upper() if m else None


# ── Version extraction ───────────────────────────────────────────────────────

def extract_semantic_versions(text: str) -> Optional[str]:
    """Extract version constraints (<, <=, >, >=) from text. Falls back to first plain version."""
    constraints: List[str] = []

    for pat, op in _RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = next((g.strip() for g in reversed(groups)
                      if _VERSION_TOKEN_RE.fullmatch(g.strip(), re.IGNORECASE)), None)
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        for t in _VERSION_TOKEN_RE.findall(text):
            cand = f"{op0}{t}"
            if cand not in constraints:
                constraints.append(cand)
        return ", ".join(_dedupe(constraints))

    upd = _UPDATE_RE.search(text)
    if upd:
        suffix = re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I)
        return f"<=Update {upd.group(1)}" if suffix else f"Update {upd.group(1)}"

    m = _VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def _clean_name(raw: str, clean_re: re.Pattern) -> str:
    s = _UPDATE_RE.sub("", raw)
    s = clean_re.sub("", s)
    s = _VERSION_TOKEN_RE.sub("", s)
    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = _TRAILING_JUNK.sub("", s)
    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    return _TRAILING_JUNK.sub("", s)


def clean_software_name(raw: str) -> str:
    return _clean_name(raw, _SW_CLEAN_RE)


def clean_hardware_name(raw: str) -> str:
    return _clean_name(raw, _HW_CLEAN_RE)


print("Shared utilities loaded.")

Shared utilities loaded.


## 3. Load Input Data

In [3]:
df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
print(f"Loaded {len(df)} rows")
df.head(3)

Loaded 15961 rows


,article_index,url,title,software_relations,hardware_relations
0,0,https://thehackernews.com/2026/02/aeternum-c2-...,Aeternum C2 Botnet Stores Encrypted Commands o...,,
1,1,https://thehackernews.com/2026/02/uat-10027-ta...,UAT-10027 Targets U.S. Education and Healthcar...,"Fondue.exe, Dohdoor; mblctr.exe, Dohdoor; Scre...",
2,2,https://thehackernews.com/2026/02/threatsday-b...,"ThreatsDay Bulletin: Kali Linux + Claude, Chro...","Apache ActiveMQ, CVE-2023-46604; WinRAR, CVE-2...",


In [4]:
def parse_relations_cell(cell: str) -> List[str]:
    """Split a semicolon-delimited relations cell into a list of strings."""
    if not cell or not cell.strip():
        return []
    return [r.strip() for r in cell.split(";") if r.strip()]


# Build parallel lists: one entry per article, preserving original order
urls               = df["url"].tolist()
titles             = df["title"].tolist()
article_indices    = df["article_index"].tolist()
software_relations = [parse_relations_cell(c) for c in df["software_relations"]]
hardware_relations = [parse_relations_cell(c) for c in df["hardware_relations"]]

print(f"Articles: {len(software_relations)}")
print(f"SW relation entries total: {sum(len(x) for x in software_relations)}")
print(f"HW relation entries total: {sum(len(x) for x in hardware_relations)}")

# Quick preview
for i, (sw, hw) in enumerate(zip(software_relations, hardware_relations)):
    if sw or hw:
        print(f"\nFirst non-empty article [{i}] — URL: {urls[i]}")
        print(" SW:", sw[:3])
        print(" HW:", hw[:3])
        break

Articles: 15961
SW relation entries total: 28468
HW relation entries total: 5541

First non-empty article [1] — URL: https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html
 SW: ['Fondue.exe, Dohdoor', 'mblctr.exe, Dohdoor', 'ScreenClippingHost.exe, Dohdoor']
 HW: []


## 4. Load Reference Datasets

In [5]:
# CVE JSON
with open(CVE_JSON_PATH, "r") as f:
    cve_records = json.load(f)
print(f"CVE records: {len(cve_records):,}")
print("Sample:", cve_records[2])

CVE records: 261,257
Sample: {'identifier': 'CVE-2024-23259', 'description': {'description_data': [{'lang': 'en', 'value': 'The issue was addressed with improved checks. This issue is fixed in iOS 16.7.6 and iPadOS 16.7.6, iOS 17.4 and iPadOS 17.4, macOS Sonoma 14.4. Processing web content may lead to a denial-of-service.'}]}, 'publishedDate': '2024-03-08T02:15Z', 'updatedDate': '2024-12-20T17:08Z', 'CVSS_score': 6.5, 'CVSS_vector': 'CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:U/C:N/I:N/A:H', 'Base_Severity': 'MEDIUM', 'Impact_Score': 3.6, 'Exploitability_Score': 2.8, 'url': ['http://seclists.org/fulldisclosure/2024/Mar/21', 'http://seclists.org/fulldisclosure/2024/Mar/21', 'https://support.apple.com/en-us/HT214081', 'https://support.apple.com/en-us/HT214081', 'https://support.apple.com/en-us/HT214082', 'https://support.apple.com/en-us/HT214082', 'https://support.apple.com/en-us/HT214084', 'https://support.apple.com/en-us/HT214084']}


In [6]:
# CISA Known Exploited Vulnerabilities
cisa_df = pd.read_csv(CISA_CSV_PATH)
cisa_df["cveID"] = cisa_df["cveID"].astype(str).str.upper().str.strip()
print(f"CISA entries: {len(cisa_df):,}")
cisa_df.head(3)

CISA entries: 1,313


,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,notes,cwes
0,CVE-2025-24813,Apache,Tomcat,Apache Tomcat Path Equivalence Vulnerability,2025-04-01,Apache Tomcat contains a path equivalence vuln...,"Apply mitigations per vendor instructions, fol...",2025-04-22,Unknown,https://lists.apache.org/thread/j5fkjv2k477os9...,"CWE-44, CWE-502"
1,CVE-2024-20439,Cisco,Smart Licensing Utility,Cisco Smart Licensing Utility Static Credentia...,2025-03-31,Cisco Smart Licensing Utility contains a stati...,"Apply mitigations per vendor instructions, fol...",2025-04-21,Unknown,https://sec.cloudapps.cisco.com/security/cente...,CWE-912
2,CVE-2025-2783,Google,Chromium Mojo,Google Chromium Mojo Sandbox Escape Vulnerability,2025-03-27,Google Chromium Mojo on Windows contains a san...,"Apply mitigations per vendor instructions, fol...",2025-04-17,Unknown,https://chromereleases.googleblog.com/2025/03/...,NaN


In [7]:
# CWE JSON
with open(CWE_JSON_PATH, "r") as f:
    cwe_records = json.load(f)
print(f"CWE records: {len(cwe_records):,}")

CWE records: 1,057


## 5. Build Lookup Indices

In [8]:
# ── Known CVE set (for A/B split) ────────────────────────────────────────────
known_cves: Set[str] = set()
for rec in cve_records:
    for cve in find_all_cves(rec.get("identifier", "")):
        known_cves.add(cve)
print(f"Known CVE IDs: {len(known_cves):,}")


# ── CISA map: CVE -> set of candidate software/hardware strings ──────────────
cisa_map: Dict[str, Set[str]] = {}
for _, row in cisa_df.iterrows():
    cve = row["cveID"]
    vals: Set[str] = set()
    for col in ("vendorProject", "product"):
        if pd.notna(row.get(col)):
            vals.add(str(row[col]))
    if pd.notna(row.get("vendorProject")) and pd.notna(row.get("product")):
        vals.add(f"{row['vendorProject']} {row['product']}")
    cisa_map.setdefault(cve, set()).update(v for v in vals if v.strip())
print(f"CISA map CVEs: {len(cisa_map):,}")


# ── CVE description map: CVE -> {desc, urls} ─────────────────────────────────
cve_desc_map: Dict[str, Dict[str, Any]] = {}
for rec in cve_records:
    m = _CVE_RE.search(rec.get("identifier", ""))
    if not m:
        continue
    cve_id = normalize_cve(m.group(1), m.group(2))
    desc = " ".join(
        d.get("value", "")
        for d in rec.get("description", {}).get("description_data", [])
        if d.get("lang", "").lower() == "en"
    )
    cve_desc_map[cve_id] = {
        "desc": desc,
        "urls": [str(u) for u in rec.get("url", []) if isinstance(u, str)],
    }
print(f"CVE desc map entries: {len(cve_desc_map):,}")

Known CVE IDs: 261,257
CISA map CVEs: 1,313
CVE desc map entries: 261,257


In [9]:
# ── CISA + CVE text corpora for Type C inverted index ────────────────────────

def _build_text_corpus(cisa_df: pd.DataFrame, cve_desc_map: Dict) -> Tuple[Dict, Dict]:
    cisa_texts: Dict[str, str] = {}
    for _, row in cisa_df.iterrows():
        cve = str(row["cveID"]).strip().upper()
        parts = [str(row[c]) for c in ("vulnerabilityName", "shortDescription", "notes", "vendorProject", "product")
                 if c in row and pd.notna(row[c])]
        cisa_texts[cve] = norm_text(" ".join(parts))

    cve_texts: Dict[str, str] = {
        cve: norm_text(info["desc"])
        for cve, info in cve_desc_map.items()
    }
    return cisa_texts, cve_texts


def _build_inverted_index(corpus: Dict[str, str]) -> Tuple[Dict, Dict]:
    inv: Dict[str, Set[str]] = defaultdict(set)
    tokens_by_id: Dict[str, Set[str]] = {}
    for cve, text in corpus.items():
        toks = set(tokenize(text))
        tokens_by_id[cve] = toks
        for t in toks:
            inv[t].add(cve)
    return dict(inv), tokens_by_id


cisa_texts, cve_texts = _build_text_corpus(cisa_df, cve_desc_map)
inv_cisa, cisa_tokens_by_cve = _build_inverted_index(cisa_texts)
inv_cve,  cve_tokens_by_cve  = _build_inverted_index(cve_texts)
print(f"CISA inverted index: {len(inv_cisa):,} tokens")
print(f"CVE  inverted index: {len(inv_cve):,} tokens")

CISA inverted index: 4,350 tokens
CVE  inverted index: 211,615 tokens


In [10]:
# ── CWE text corpus + inverted index ─────────────────────────────────────────

cwe_texts: Dict[str, str] = {}
inv_cwe: Dict[str, Set[str]] = defaultdict(set)

for rec in cwe_records:
    cwe_id = f"CWE-{rec.get('identifier', '')}".strip()
    if cwe_id == "CWE-":
        continue
    joined = " ".join(str(rec.get(k) or "") for k in ("name", "description", "Alternate_Terms"))
    text = norm_text(joined)
    cwe_texts[cwe_id] = text
    for tok in set(tokenize(text)):
        inv_cwe[tok].add(cwe_id)

inv_cwe = dict(inv_cwe)
print(f"CWE entries: {len(cwe_texts):,}")

CWE entries: 1,057


## 6. Classification Functions

In [11]:
# ── Helpers: parse a single 'Entity, Vulnerability' string ───────────────────

def split_entry(entry: str) -> Tuple[str, str]:
    """Return (entity_part, vuln_part). vuln_part may be empty string."""
    parts = entry.split(",", 1)
    return parts[0].strip(), (parts[1].strip() if len(parts) > 1 else "")


# ── Type A / B: does pair appear in CISA or CVE JSON? ────────────────────────

def _pair_in_cisa(entity: str, cve: str) -> bool:
    if cve not in cisa_map:
        return False
    for cand in cisa_map[cve]:
        if token_subset(entity, cand) or token_subset(cand, entity):
            return True
        if fuzzy_match(entity, cand, threshold=88):
            return True
    return False


def _pair_in_cve_json(entity: str, cve: str) -> bool:
    info = cve_desc_map.get(cve)
    if not info:
        return False
    desc = info.get("desc", "")
    if token_subset(entity, desc):
        return True
    if any(token_subset(entity, u) for u in info.get("urls", [])):
        return True
    if fuzzy_match(entity, desc, threshold=90):
        return True
    return False


# ── Type C: match NL vulnerability text to a CVE ─────────────────────────────

_NL_MATCH_CACHE: Dict[str, Optional[str]] = {}

def _match_nl_to_cve(nl_text: str, max_candidates: int = 80) -> Optional[str]:
    key = nl_text.strip().lower()
    if key in _NL_MATCH_CACHE:
        return _NL_MATCH_CACHE[key]

    toks = set(tokenize(nl_text))
    if not toks:
        _NL_MATCH_CACHE[key] = None
        return None

    # Candidate generation via union of postings lists
    cands: Set[str] = set()
    for t in toks:
        cands |= inv_cisa.get(t, set())
        cands |= inv_cve.get(t, set())

    if not cands:
        _NL_MATCH_CACHE[key] = None
        return None

    # Score by token overlap, prune to top-K
    scores = [
        (cve, len(toks & (cisa_tokens_by_cve.get(cve) or cve_tokens_by_cve.get(cve, set()))))
        for cve in cands
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    top_k = [cve for cve, s in scores[:max_candidates] if s > 0]

    # Heavy check on pruned set
    nl_norm = norm_text(nl_text)
    for cve in top_k:
        base = cisa_texts.get(cve) or cve_texts.get(cve, "")
        if not base:
            continue
        nl_set = set(nl_norm.split())
        if nl_norm in base or (nl_set and nl_set.issubset(set(base.split()))):
            _NL_MATCH_CACHE[key] = cve
            return cve
        if HAVE_RAPIDFUZZ and fuzz.partial_ratio(nl_norm, base) >= 85:
            _NL_MATCH_CACHE[key] = cve
            return cve

    _NL_MATCH_CACHE[key] = None
    return None


# ── Type D: match NL vulnerability text to a CWE ─────────────────────────────

def _match_nl_to_cwe(nl_text: str) -> Optional[str]:
    toks = set(tokenize(nl_text))
    if not toks:
        return None

    cands: Set[str] = set()
    for t in toks:
        cands |= inv_cwe.get(t, set())

    if not cands:
        return None

    scores = [
        (cwe, len(toks & set(tokenize(cwe_texts[cwe]))))
        for cwe in cands
    ]
    scores.sort(key=lambda x: x[1], reverse=True)

    nl_norm = norm_text(nl_text)
    for cwe, _ in scores[:50]:
        text = cwe_texts[cwe]
        nl_set = set(nl_norm.split())
        if nl_norm in text or (nl_set and nl_set.issubset(set(text.split()))):
            return cwe
        if HAVE_RAPIDFUZZ and fuzz.partial_ratio(nl_norm, text) >= 85:
            return cwe
    return None


print("Classification functions ready.")

Classification functions ready.


## 7. Run Classification Pipeline

In [12]:
def classify_relations(
    relations_per_article: List[List[str]],
    entity_type: str,         # "software" or "hardware"
) -> Dict[str, List[List[Any]]]:
    """
    Classify all relations into Types A–E.
    Returns a dict with keys type_a … type_e, each a list-of-lists.
    Each list corresponds to one article and preserves the original order.
    """
    assert entity_type in ("software", "hardware")

    type_a: List[List[str]] = []
    type_b: List[List[str]] = []
    type_c: List[List[Tuple[str, str]]] = []   # [(original_entry, matched_CVE)]
    type_d: List[List[Tuple[str, str]]] = []   # [(original_entry, matched_CWE)]
    type_e: List[List[str]] = []

    for sub in tqdm(relations_per_article, desc=f"Classifying {entity_type}"):
        a, b, c, d, e = [], [], [], [], []

        for entry in sub:
            entity, vuln_text = split_entry(entry)
            cve_in_text = extract_cve(vuln_text)

            if cve_in_text:
                # CVE branch → Type A or B
                in_sources = (
                    _pair_in_cisa(entity, cve_in_text) or
                    _pair_in_cve_json(entity, cve_in_text)
                )
                if in_sources:
                    a.append(entry)
                else:
                    b.append(entry)
            else:
                # Natural language branch → Type C, D, or E
                matched_cve = _match_nl_to_cve(vuln_text)
                if matched_cve:
                    c.append((entry, matched_cve))
                else:
                    matched_cwe = _match_nl_to_cwe(vuln_text)
                    if matched_cwe:
                        d.append((entry, matched_cwe))
                    else:
                        e.append(entry)

        type_a.append(a)
        type_b.append(b)
        type_c.append(c)
        type_d.append(d)
        type_e.append(e)

    return {
        f"type_a_{entity_type}": type_a,
        f"type_b_{entity_type}": type_b,
        f"type_c_{entity_type}": type_c,
        f"type_d_{entity_type}": type_d,
        f"type_e_{entity_type}": type_e,
    }


sw_types = classify_relations(software_relations, "software")
hw_types = classify_relations(hardware_relations, "hardware")

Classifying software:   0%|          | 0/15961 [00:00<?, ?it/s]

Classifying hardware:   0%|          | 0/15961 [00:00<?, ?it/s]

In [13]:
# Quick count summary
all_types = {**sw_types, **hw_types}

print(f"{'Type':<30} {'Non-empty articles':>20} {'Total entries':>15}")
print("-" * 68)
for k, v in all_types.items():
    non_empty = sum(1 for x in v if x)
    total = sum(len(x) for x in v)
    print(f"{k:<30} {non_empty:>20} {total:>15}")

Type                             Non-empty articles   Total entries
--------------------------------------------------------------------
type_a_software                                1846            6241
type_b_software                                1556            6541
type_c_software                                1656            3279
type_d_software                                   6               6
type_e_software                                4209           12401
type_a_hardware                                 249             709
type_b_hardware                                 424            2184
type_c_hardware                                 353             765
type_d_hardware                                   2               2
type_e_hardware                                 675            1881


## 8. Convert to Structured JSON (with URLs)

In [14]:
# ── Shared structured-output builders ────────────────────────────────────────

def _make_sw_record(
    entity_raw: str,
    vuln: Optional[str],
    vuln_mention: Optional[str],
    url: Optional[str],
) -> Dict[str, Optional[str]]:
    version = extract_semantic_versions(entity_raw)
    name = clean_software_name(entity_raw)
    return {
        "software": name or entity_raw.strip(),
        "software_version": version,
        "vulnerability_mention": vuln_mention,
        "vulnerability": vuln,
        "article_url": url,
    }


def _make_hw_record(
    entity_raw: str,
    vuln: Optional[str],
    vuln_mention: Optional[str],
    url: Optional[str],
) -> Dict[str, Optional[str]]:
    version = extract_semantic_versions(entity_raw)
    name = clean_hardware_name(entity_raw)
    return {
        "hardware": name or entity_raw.strip(),
        "hardware_version": version,
        "vulnerability_mention": vuln_mention,
        "vulnerability": vuln,
        "article_url": url,
    }


# ── Converters per type ───────────────────────────────────────────────────────

def convert_type_a_b(
    data: List[List[str]],
    urls: List[str],
    entity_type: str,
) -> List[List[Dict]]:
    """Type A/B: entry is 'Entity, CVE-XXXX-YYYY'. CVE is the vulnerability."""
    make = _make_sw_record if entity_type == "software" else _make_hw_record
    out = []
    for sub, url in zip(data, urls):
        if not sub:
            out.append([])
            continue
        records = []
        for entry in sub:
            entity_raw, vuln_text = split_entry(entry)
            cve = extract_cve(vuln_text)
            records.append(make(entity_raw, cve or vuln_text or None, None, url or None))
        out.append(records)
    return out


def convert_type_c(
    data: List[List[Tuple[str, str]]],
    urls: List[str],
    entity_type: str,
) -> List[List[Dict]]:
    """Type C: entry is (original_string, matched_CVE). original_string has 'Entity, NL mention'."""
    make = _make_sw_record if entity_type == "software" else _make_hw_record
    out = []
    for sub, url in zip(data, urls):
        if not sub:
            out.append([])
            continue
        records = []
        for item in sub:
            if not (isinstance(item, (list, tuple)) and len(item) == 2):
                continue
            original, matched_cve = item
            entity_raw, mention = split_entry(str(original))
            records.append(make(entity_raw, matched_cve or None, mention or None, url or None))
        out.append(records)
    return out


def convert_type_d(
    data: List[List[Tuple[str, str]]],
    urls: List[str],
    entity_type: str,
) -> List[List[Dict]]:
    """Type D: entry is (original_string, matched_CWE). original_string has 'Entity, NL mention'."""
    make = _make_sw_record if entity_type == "software" else _make_hw_record
    out = []
    for sub, url in zip(data, urls):
        if not sub:
            out.append([])
            continue
        records = []
        for item in sub:
            if not (isinstance(item, (list, tuple)) and len(item) == 2):
                continue
            original, matched_cwe = item
            entity_raw, mention = split_entry(str(original))
            records.append(make(entity_raw, matched_cwe or None, mention or None, url or None))
        out.append(records)
    return out


def convert_type_e(
    data: List[List[str]],
    urls: List[str],
    entity_type: str,
) -> List[List[Dict]]:
    """Type E: entry is 'Entity, NL mention' with no matched CVE or CWE."""
    make = _make_sw_record if entity_type == "software" else _make_hw_record
    out = []
    for sub, url in zip(data, urls):
        if not sub:
            out.append([])
            continue
        records = []
        for entry in sub:
            entity_raw, mention = split_entry(entry)
            records.append(make(entity_raw, None, mention or None, url or None))
        out.append(records)
    return out


print("Converters defined.")

Converters defined.


In [15]:
# Run all conversions

combined = {
    "software_type_a": convert_type_a_b(sw_types["type_a_software"], urls, "software"),
    "software_type_b": convert_type_a_b(sw_types["type_b_software"], urls, "software"),
    "software_type_c": convert_type_c  (sw_types["type_c_software"], urls, "software"),
    "software_type_d": convert_type_d  (sw_types["type_d_software"], urls, "software"),
    "software_type_e": convert_type_e  (sw_types["type_e_software"], urls, "software"),

    "hardware_type_a": convert_type_a_b(hw_types["type_a_hardware"], urls, "hardware"),
    "hardware_type_b": convert_type_a_b(hw_types["type_b_hardware"], urls, "hardware"),
    "hardware_type_c": convert_type_c  (hw_types["type_c_hardware"], urls, "hardware"),
    "hardware_type_d": convert_type_d  (hw_types["type_d_hardware"], urls, "hardware"),
    "hardware_type_e": convert_type_e  (hw_types["type_e_hardware"], urls, "hardware"),
}

# Final count summary
print(f"\n{'Type':<25} {'Entries':>10}")
print("-" * 37)
for k, v in combined.items():
    total = sum(len(x) for x in v)
    print(f"{k:<25} {total:>10}")


Type                         Entries
-------------------------------------
software_type_a                 6241
software_type_b                 6541
software_type_c                 3279
software_type_d                    6
software_type_e                12401
hardware_type_a                  709
hardware_type_b                 2184
hardware_type_c                  765
hardware_type_d                    2
hardware_type_e                 1881


## 9. Inspect Samples

In [16]:
# Helper to print first N non-empty articles for a given type
def preview(type_key: str, n: int = 3):
    data = combined[type_key]
    shown = 0
    for i, records in enumerate(data):
        if records:
            print(f"  Article {i} | {records[0].get('article_url', '')}")
            for r in records[:2]:
                print("   ", r)
            shown += 1
        if shown >= n:
            break

print("=== software_type_a (known CVE relations) ===")
preview("software_type_a")

=== software_type_a (known CVE relations) ===
  Article 2 | https://thehackernews.com/2026/02/threatsday-bulletin-kali-linux-claude.html
    {'software': 'Apache ActiveMQ', 'software_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2023-46604', 'article_url': 'https://thehackernews.com/2026/02/threatsday-bulletin-kali-linux-claude.html'}
  Article 6 | https://thehackernews.com/2026/02/cisco-sd-wan-zero-day-cve-2026-20127.html
    {'software': 'Cisco SD-WAN Software', 'software_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2022-20775', 'article_url': 'https://thehackernews.com/2026/02/cisco-sd-wan-zero-day-cve-2026-20127.html'}
  Article 15 | https://thehackernews.com/2026/02/sshstalker-botnet-uses-irc-c2-to.html
    {'software': 'Linux kernel .x', 'software_version': '2.6', 'vulnerability_mention': None, 'vulnerability': 'CVE-2009-2692', 'article_url': 'https://thehackernews.com/2026/02/sshstalker-botnet-uses-irc-c2-to.html'}
    {'software': 

In [17]:
print("=== software_type_b (new CVE relations) ===")
preview("software_type_b")

=== software_type_b (new CVE relations) ===
  Article 2 | https://thehackernews.com/2026/02/threatsday-bulletin-kali-linux-claude.html
    {'software': 'WinRAR', 'software_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2025-8088', 'article_url': 'https://thehackernews.com/2026/02/threatsday-bulletin-kali-linux-claude.html'}
    {'software': 'strongMan', 'software_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2026-25998', 'article_url': 'https://thehackernews.com/2026/02/threatsday-bulletin-kali-linux-claude.html'}
  Article 6 | https://thehackernews.com/2026/02/cisco-sd-wan-zero-day-cve-2026-20127.html
    {'software': 'Cisco Catalyst SD-WAN Controller', 'software_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2026-20127', 'article_url': 'https://thehackernews.com/2026/02/cisco-sd-wan-zero-day-cve-2026-20127.html'}
    {'software': 'Cisco Catalyst SD-WAN Manager', 'software_version': None, 'vulnerability_mention': None,

In [18]:
print("=== software_type_c (NL -> CVE matched) ===")
preview("software_type_c")

=== software_type_c (NL -> CVE matched) ===
  Article 1 | https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html
    {'software': 'propsys.dll', 'software_version': None, 'vulnerability_mention': 'propsys.dll', 'vulnerability': 'CVE-2016-4349', 'article_url': 'https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html'}
    {'software': 'Cobalt Strike Beacon', 'software_version': None, 'vulnerability_mention': 'Cobalt Strike Beacon', 'vulnerability': 'CVE-2022-39197', 'article_url': 'https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html'}
  Article 4 | https://thehackernews.com/2026/02/fake-nextjs-repos-target-developers.html
    {'software': 'Node.js', 'software_version': None, 'vulnerability_mention': 'eslint-validator', 'vulnerability': 'CVE-2011-3020', 'article_url': 'https://thehackernews.com/2026/02/fake-nextjs-repos-target-developers.html'}
  Article 20 | https://thehackernews.com/2026/02/malicious-chrome-extensions-caught.ht

In [19]:
print("=== software_type_d (NL -> CWE matched) ===")
preview("software_type_d")

=== software_type_d (NL -> CWE matched) ===
  Article 1562 | https://thehackernews.com/2025/04/pakistan-linked-hackers-expand-targets.html
    {'software': 'Firefox', 'software_version': None, 'vulnerability_mention': 'Action RAT', 'vulnerability': 'CWE-119', 'article_url': 'https://thehackernews.com/2025/04/pakistan-linked-hackers-expand-targets.html'}
  Article 3426 | https://thehackernews.com/2024/03/researchers-highlight-googles-gemini-ai.html
    {'software': 'Gemini', 'software_version': None, 'vulnerability_mention': 'synonym attack', 'vulnerability': 'CWE-1007', 'article_url': 'https://thehackernews.com/2024/03/researchers-highlight-googles-gemini-ai.html'}
  Article 4825 | https://thehackernews.com/2023/05/sidecopy-using-action-rat-and-allakore.html
    {'software': 'Action RAT', 'software_version': None, 'vulnerability_mention': 'Action RAT', 'vulnerability': 'CWE-119', 'article_url': 'https://thehackernews.com/2023/05/sidecopy-using-action-rat-and-allakore.html'}


In [20]:
print("=== software_type_e (unmatched NL) ===")
preview("software_type_e")

=== software_type_e (unmatched NL) ===
  Article 1 | https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html
    {'software': 'Fondue.exe', 'software_version': None, 'vulnerability_mention': 'Dohdoor', 'vulnerability': None, 'article_url': 'https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html'}
    {'software': 'mblctr.exe', 'software_version': None, 'vulnerability_mention': 'Dohdoor', 'vulnerability': None, 'article_url': 'https://thehackernews.com/2026/02/uat-10027-targets-us-education-and.html'}
  Article 4 | https://thehackernews.com/2026/02/fake-nextjs-repos-target-developers.html
    {'software': 'Visual Studio Code', 'software_version': None, 'vulnerability_mention': 'Contagious Interview', 'vulnerability': None, 'article_url': 'https://thehackernews.com/2026/02/fake-nextjs-repos-target-developers.html'}
    {'software': 'Node.js', 'software_version': None, 'vulnerability_mention': 'Contagious Interview', 'vulnerability': None, 'article_url

In [21]:
print("=== hardware_type_a ===")
preview("hardware_type_a")

=== hardware_type_a ===
  Article 360 | https://thehackernews.com/2025/12/amazon-exposes-years-long-gru-cyber.html
    {'hardware': 'WatchGuard Firebox', 'hardware_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2022-26318', 'article_url': 'https://thehackernews.com/2025/12/amazon-exposes-years-long-gru-cyber.html'}
    {'hardware': 'WatchGuard XTM', 'hardware_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2022-26318', 'article_url': 'https://thehackernews.com/2025/12/amazon-exposes-years-long-gru-cyber.html'}
  Article 379 | https://thehackernews.com/2025/12/threatsday-bulletin-spyware-alerts.html
    {'hardware': 'TBK DVR', 'hardware_version': None, 'vulnerability_mention': None, 'vulnerability': 'CVE-2024-3721', 'article_url': 'https://thehackernews.com/2025/12/threatsday-bulletin-spyware-alerts.html'}
  Article 588 | https://thehackernews.com/2025/10/experts-reports-sharp-increase-in.html
    {'hardware': 'TBK DVR-4104', 'hardware_version'

## 10. Save Output

In [22]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(combined, f, indent=2, ensure_ascii=False)

print(f"Saved to {OUTPUT_PATH}")

Saved to classification_types_in_json.json


## 11. (Optional) Flat Export — one row per relation

Useful for quick inspection or downstream processing.

In [23]:
rows = []
for type_key, records_per_article in combined.items():
    entity_type, _, type_letter = type_key.split("_")  # e.g. software, type, a
    for article_i, records in enumerate(records_per_article):
        for rec in records:
            rows.append({
                "classification": type_key,
                "entity_type": entity_type,
                "type_letter": type_letter.upper(),
                "article_index": article_indices[article_i],
                "article_url": rec.get("article_url"),
                "entity": rec.get("software") or rec.get("hardware"),
                "entity_version": rec.get("software_version") or rec.get("hardware_version"),
                "vulnerability": rec.get("vulnerability"),
                "vulnerability_mention": rec.get("vulnerability_mention"),
            })

flat_df = pd.DataFrame(rows)
flat_df.to_csv("classification_flat.csv", index=False)
print(f"Flat export: {len(flat_df):,} rows")
flat_df.head(10)

Flat export: 34,009 rows


,classification,entity_type,type_letter,article_index,article_url,entity,entity_version,vulnerability,vulnerability_mention
0,software_type_a,software,A,2,https://thehackernews.com/2026/02/threatsday-b...,Apache ActiveMQ,None,CVE-2023-46604,None
1,software_type_a,software,A,6,https://thehackernews.com/2026/02/cisco-sd-wan...,Cisco SD-WAN Software,None,CVE-2022-20775,None
2,software_type_a,software,A,15,https://thehackernews.com/2026/02/sshstalker-b...,Linux kernel .x,2.6,CVE-2009-2692,None
3,software_type_a,software,A,15,https://thehackernews.com/2026/02/sshstalker-b...,Linux kernel .x,2.6,CVE-2010-1173,None
4,software_type_a,software,A,15,https://thehackernews.com/2026/02/sshstalker-b...,Linux kernel .x,2.6,CVE-2009-2908,None
5,software_type_a,software,A,15,https://thehackernews.com/2026/02/sshstalker-b...,Linux kernel .x,2.6,CVE-2010-2959,None
6,software_type_a,software,A,43,https://thehackernews.com/2026/02/wormable-xmr...,WinRing0x64.sys,None,CVE-2020-14979,None
7,software_type_a,software,A,48,https://thehackernews.com/2026/02/ai-assisted-...,Veeam Backup & Replication,None,CVE-2023-27532,None
8,software_type_a,software,A,48,https://thehackernews.com/2026/02/ai-assisted-...,Veeam Backup & Replication,None,CVE-2024-40711,None
9,software_type_a,software,A,62,https://thehackernews.com/2026/02/threatsday-b...,GitLab,None,CVE-2021-22175,None
